# EDA 008: Clean pipeline — iterate cleaned chunks and write to Parquet

Experiment with the load/clean and export steps without running the full script.
Use a small raw CSV or small chunksize for quick runs.

## Config

In [3]:
!pwd

/home/ryanr/workspace/steam_recommendations/notebooks/eda


In [4]:
from pathlib import Path

input_path = Path("../../data/raw/steam_reviews_full.csv")
output_path = Path("../../data/interim/steam_reviews_cleaned.parquet")
chunksize = 100_000
language = "english"
columns = None

print("input_path:", input_path)
print("output_path:", output_path)
print("chunksize:", chunksize)

input_path: ../../data/raw/steam_reviews_full.csv
output_path: ../../data/interim/steam_reviews_cleaned.parquet
chunksize: 100000


# SAMPLE SIZE

### Load + clean only: inspect first N chunks

In [5]:
from steam_review_ml.data.loaders import iter_clean_chunks

N_CHUNKS = 3  # consume only first N chunks for quick inspection
chunks = []
for i, df in enumerate(iter_clean_chunks(input_path, chunksize, language, columns)):
    chunks.append(df)
    if i >= N_CHUNKS - 1:
        break

print("Number of chunks:", len(chunks))
if chunks:
    print("First chunk shape:", chunks[0].shape)
    print("Columns:", list(chunks[0].columns))
    display(chunks[0].head())

Number of chunks: 3
First chunk shape: (23666, 20)
Columns: ['review_id', 'app_id', 'app_name', 'author.steamid', 'review', 'recommended', 'votes_helpful', 'author.num_games_owned', 'author.num_reviews', 'author.playtime_last_two_weeks', 'author.playtime_at_review', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'timestamp_created', 'author.last_played', 'is_helpful', 'review_length_chars', 'author_playtime_last_two_weeks_missing', 'author_playtime_at_review_missing']


,review_id,app_id,app_name,author.steamid,review,recommended,votes_helpful,author.num_games_owned,author.num_reviews,author.playtime_last_two_weeks,author.playtime_at_review,steam_purchase,received_for_free,written_during_early_access,timestamp_created,author.last_played,is_helpful,review_length_chars,author_playtime_last_two_weeks_missing,author_playtime_at_review_missing
3,85184605,292030,The Witcher 3: Wild Hunt,76561199054755373,"One of the best RPG's of all time, worthy of a...",True,0,5,3,3200.0,5524.0,True,False,False,1611379970,1.611384e+09,False,59,False,False
5,85184171,292030,The Witcher 3: Wild Hunt,76561198170193529,"good story, good graphics. lots to do.",True,0,11,1,823.0,823.0,True,False,False,1611379264,1.611379e+09,False,38,False,False
6,85184064,292030,The Witcher 3: Wild Hunt,76561198119302812,"dis gud,",True,0,27,2,3398.0,4192.0,True,False,False,1611379091,1.611352e+09,False,8,False,False
18,85180436,292030,The Witcher 3: Wild Hunt,76561198065591528,favorite game of all time cant wait for the Ne...,True,0,33,1,177.0,23329.0,True,False,False,1611373086,1.611219e+09,False,59,False,False
20,85179753,292030,The Witcher 3: Wild Hunt,76561198996835044,Why wouldn't you get this,True,0,131,2,2004.0,8557.0,True,False,False,1611371978,1.611371e+09,False,25,False,False


### Export: write to Parquet

In [6]:
from steam_review_ml.data.export import write_parquet_chunked

# Option A: write the chunks we already collected (quick test)
write_parquet_chunked(iter(chunks), output_path)
print("Wrote:", output_path)

# Read back and verify
import pandas as pd

df_out = pd.read_parquet(output_path)
print("Read back shape:", df_out.shape)
display(df_out.head())

Wrote: ../../data/interim/steam_reviews_cleaned.parquet
Read back shape: (82816, 20)


,review_id,app_id,app_name,author.steamid,review,recommended,votes_helpful,author.num_games_owned,author.num_reviews,author.playtime_last_two_weeks,author.playtime_at_review,steam_purchase,received_for_free,written_during_early_access,timestamp_created,author.last_played,is_helpful,review_length_chars,author_playtime_last_two_weeks_missing,author_playtime_at_review_missing
0,85184605,292030,The Witcher 3: Wild Hunt,76561199054755373,"One of the best RPG's of all time, worthy of a...",True,0,5,3,3200.0,5524.0,True,False,False,1611379970,1.611384e+09,False,59,False,False
1,85184171,292030,The Witcher 3: Wild Hunt,76561198170193529,"good story, good graphics. lots to do.",True,0,11,1,823.0,823.0,True,False,False,1611379264,1.611379e+09,False,38,False,False
2,85184064,292030,The Witcher 3: Wild Hunt,76561198119302812,"dis gud,",True,0,27,2,3398.0,4192.0,True,False,False,1611379091,1.611352e+09,False,8,False,False
3,85180436,292030,The Witcher 3: Wild Hunt,76561198065591528,favorite game of all time cant wait for the Ne...,True,0,33,1,177.0,23329.0,True,False,False,1611373086,1.611219e+09,False,59,False,False
4,85179753,292030,The Witcher 3: Wild Hunt,76561198996835044,Why wouldn't you get this,True,0,131,2,2004.0,8557.0,True,False,False,1611371978,1.611371e+09,False,25,False,False


# Full Export Run

In [9]:
from pathlib import Path
import pandas as pd
from steam_review_ml.data.loaders import iter_clean_chunks
from steam_review_ml.data.export import write_parquet_chunked

input_path = Path("../../data/raw/steam_reviews_full.csv")
output_path = Path("../../data/interim/steam_reviews_cleaned_FULL_test.parquet")
chunksize = 100_000
language = "english"
columns = None

print("input_path:", input_path)
print("output_path:", output_path)
print("chunksize:", chunksize)

input_path: ../../data/raw/steam_reviews_full.csv
output_path: ../../data/interim/steam_reviews_cleaned_FULL_test.parquet
chunksize: 100000


In [10]:
# Full run: stream all, write to Parquet (one chunk in memory at a time)
chunks_full = iter_clean_chunks(input_path, chunksize, language, columns)
write_parquet_chunked(chunks_full, output_path)
print("Wrote full Parquet:", output_path)

Wrote full Parquet: ../../data/interim/steam_reviews_cleaned_FULL_test.parquet


In [11]:
# Read the saved Parquet and show first 10 records
df_full = pd.read_parquet(output_path)
print("Total rows:", len(df_full))
display(df_full.head(10))

Total rows: 9160494


,review_id,app_id,app_name,author.steamid,review,recommended,votes_helpful,author.num_games_owned,author.num_reviews,author.playtime_last_two_weeks,author.playtime_at_review,steam_purchase,received_for_free,written_during_early_access,timestamp_created,author.last_played,is_helpful,review_length_chars,author_playtime_last_two_weeks_missing,author_playtime_at_review_missing
0,85184605,292030,The Witcher 3: Wild Hunt,76561199054755373,"One of the best RPG's of all time, worthy of a...",True,0,5,3,3200.0,5524.0,True,False,False,1611379970,1.611384e+09,False,59,False,False
1,85184171,292030,The Witcher 3: Wild Hunt,76561198170193529,"good story, good graphics. lots to do.",True,0,11,1,823.0,823.0,True,False,False,1611379264,1.611379e+09,False,38,False,False
2,85184064,292030,The Witcher 3: Wild Hunt,76561198119302812,"dis gud,",True,0,27,2,3398.0,4192.0,True,False,False,1611379091,1.611352e+09,False,8,False,False
3,85180436,292030,The Witcher 3: Wild Hunt,76561198065591528,favorite game of all time cant wait for the Ne...,True,0,33,1,177.0,23329.0,True,False,False,1611373086,1.611219e+09,False,59,False,False
4,85179753,292030,The Witcher 3: Wild Hunt,76561198996835044,Why wouldn't you get this,True,0,131,2,2004.0,8557.0,True,False,False,1611371978,1.611371e+09,False,25,False,False
5,85179400,292030,The Witcher 3: Wild Hunt,76561198284845223,it is ok\n,True,0,60,9,242.0,2518.0,True,False,False,1611371392,1.611371e+09,False,9,False,False
6,85179341,292030,The Witcher 3: Wild Hunt,76561198370568524,worth\n,True,0,59,5,35.0,517.0,True,False,False,1611371318,1.611374e+09,False,6,False,False
7,85178164,292030,The Witcher 3: Wild Hunt,76561198040150323,Isn't Geralt hot enough to get both Yennefer a...,True,0,51,37,0.0,165.0,True,False,False,1611369478,1.437876e+09,False,98,False,False
8,85177892,292030,The Witcher 3: Wild Hunt,76561198040190687,"Very Fun, Would play again!",True,0,54,1,75.0,20092.0,True,False,False,1611369121,1.611374e+09,False,27,False,False
9,85174926,292030,The Witcher 3: Wild Hunt,76561198020027165,The game is enjoyable enough but...\n-Combat h...,True,0,208,105,370.0,398.0,True,False,False,1611364401,1.611370e+09,False,491,False,False


In [12]:
df_full.shape

(9160494, 20)

In [13]:
# Count of duplicate review_ids in df_full
duplicate_count = df_full.duplicated(subset="review_id").sum()
print(f"Number of duplicate review_id values in df_full: {duplicate_count}")

Number of duplicate review_id values in df_full: 0


## Optional: compare with export_cleaned_reviews

In [14]:
# Uncomment to run the convenience wrapper on same config and compare output
# from steam_review_ml.data.export import export_cleaned_reviews
# out_alt = output_path.with_stem(output_path.stem + "_full")
# export_cleaned_reviews(input_path, out_alt, chunksize, language, columns)
# df_full = pd.read_parquet(out_alt)
# print("Full run rows:", len(df_full))